# Clasificación de SMS con una GRU

Proyecto académico con el conjunto **SMS Spam Collection** de UCI. El archivo fuente tiene dos campos separados por tabulador: etiqueta (`ham` o `spam`) y mensaje de texto. La etiqueta `ham` representa SMS legítimo. UCI describe el corpus como un conjunto público recopilado para investigación de spam móvil y lo distribuye con licencia CC BY 4.0.

El notebook carga y explora los datos, crea particiones estratificadas aproximadas 80/10/10, entrena una red GRU con TensorFlow/Keras, evalúa únicamente al final sobre prueba y produce gráficos, matriz de confusión y ejemplos de predicción medidos. La capa `TextVectorization` se adapta solo con entrenamiento y se guarda dentro del modelo.

**Sin resultados precargados:** las cifras y predicciones aparecen únicamente después de ejecutar las celdas con el corpus.

Fuente y cita: Almeida, T. & Hidalgo, J. (2011). [SMS Spam Collection, UCI Machine Learning Repository](https://doi.org/10.24432/C5CC84). Licencia CC BY 4.0. [Página del conjunto](https://archive.ics.uci.edu/dataset/228/sms+spam+collection).

## 1. Preparar Colab
Sube `train.py` y `requirements.txt` desde este proyecto cuando aparezca el selector de archivos. La canalización descargará el dataset desde UCI. Si UCI no está disponible desde el runtime, descarga `SMSSpamCollection` desde la página del conjunto y súbelo en la celda de alternativa.

In [ ]:
from google.colab import files
uploaded = files.upload()  # Selecciona train.py y requirements.txt
required = {'train.py', 'requirements.txt'}
missing = required - set(uploaded)
if missing:
    raise FileNotFoundError(f'Faltan archivos del proyecto: {sorted(missing)}')

In [ ]:
%pip install -q -r requirements.txt

## 2. Cargar, limpiar y explorar
La limpieza recorta espacios, convierte etiquetas a minúscula y descarta mensajes vacíos. No elimina palabras, números ni puntuación manualmente: la normalización de texto queda en la capa del modelo. El análisis imprime conteos y estadísticos calculados desde el archivo leído.

In [ ]:
import pandas as pd
from IPython.display import display, Image
from train import (load_dataset, clean_dataset, explore_dataset, split_dataset,
                   set_random_seeds, run_training)

set_random_seeds(42)
data_path = None
try:
    raw_df = load_dataset()
except RuntimeError as download_error:
    print(download_error)
    local_upload = files.upload()  # Sube el archivo SMSSpamCollection descargado de UCI
    candidates = [name for name in local_upload if name not in {'train.py', 'requirements.txt'}]
    if not candidates:
        raise FileNotFoundError('No se subió SMSSpamCollection.')
    data_path = candidates[0]
    raw_df = load_dataset(data_path=data_path)
df = clean_dataset(raw_df)
eda_summary = explore_dataset(df, 'reports')
display(pd.DataFrame(eda_summary['class_counts'].items(), columns=['etiqueta', 'cantidad']))
display(df.groupby('label')['message_length'].describe().round(2))
display(Image(filename='reports/dataset_exploration.png'))

## 3. Partición estratificada
Se usa semilla 42 y dos cortes estratificados. La prueba no interviene en adaptación del vocabulario, entrenamiento, selección de épocas ni umbral; el umbral del reporte base es 0.50.

In [ ]:
train_df, validation_df, test_df = split_dataset(df, seed=42)
split_table = pd.DataFrame({
    'partición': ['entrenamiento', 'validación', 'prueba'],
    'ejemplos': [len(train_df), len(validation_df), len(test_df)],
    'proporción': [len(train_df)/len(df), len(validation_df)/len(df), len(test_df)/len(df)],
})
display(split_table)
display(pd.concat([
    train_df['label'].value_counts(normalize=True).rename('entrenamiento'),
    validation_df['label'].value_counts(normalize=True).rename('validación'),
    test_df['label'].value_counts(normalize=True).rename('prueba'),
], axis=1).fillna(0).round(3))

## 4. Entrenar y evaluar
El script compartido vuelve a cargar los datos para que también pueda ejecutarse localmente desde la terminal. Entrena con particiones estratificadas, ajusta el vocabulario usando solo entrenamiento, guarda `models/sms_spam_gru.keras` y calcula los reportes desde las predicciones reales de prueba.

In [ ]:
result = run_training(data_path=data_path, epochs=20, batch_size=32)
display(pd.DataFrame([result['metrics']]).drop(columns=['classification_report', 'confusion_matrix_labels_ham_spam']))
display(pd.DataFrame(result['metrics']['classification_report']).T)
display(Image(filename='reports/training_curves.png'))
display(Image(filename='reports/confusion_matrix.png'))
predictions = pd.read_csv('reports/test_predictions.csv')
display(predictions)  # Hasta diez ejemplos generados a partir de la partición de prueba
display(pd.DataFrame([result['run_info']]))

## 5. Exportar modelo y resultados reales
Monta Google Drive y copia el modelo entrenado a `MyDrive/rnn-sms-spam/models/` y todos los archivos reales de `reports/` a `MyDrive/rnn-sms-spam/reports/`. Las carpetas se crean si no existen. La app consume texto crudo; no hace limpieza ni vectorización propia.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import shutil

local_model = Path('models/sms_spam_gru.keras')
local_reports = Path('reports')
if not local_model.is_file():
    raise FileNotFoundError('No se encontró el modelo entrenado; ejecuta primero la celda de entrenamiento.')
if not local_reports.is_dir():
    raise FileNotFoundError('No se encontró reports/; ejecuta primero la evaluación del notebook.')
report_files = sorted(path for path in local_reports.rglob('*') if path.is_file())
if not report_files:
    raise FileNotFoundError('reports/ no contiene archivos de resultados generados.')
drive_root = Path('/content/drive/MyDrive/rnn-sms-spam')
drive_models = drive_root / 'models'
drive_reports = drive_root / 'reports'
drive_models.mkdir(parents=True, exist_ok=True)
drive_reports.mkdir(parents=True, exist_ok=True)
model_destination = drive_models / local_model.name
shutil.copy2(local_model, model_destination)
copied_reports = []
for source in report_files:
    destination = drive_reports / source.relative_to(local_reports)
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)
    copied_reports.append(destination)
print(f'Modelo guardado: {model_destination}')
print('Reportes guardados:')
for destination in copied_reports:
    print(f'- {destination}')